# Connectors and disaggregation

## Learning goals
- Understand what the OmniConnector transports and why
- Compare shared-memory vs Mooncake transport with real numbers
- See why connector overhead is negligible next to inference

> **Mac track.** Every executed cell below is a pure-Python simulation that runs on any Mac with no GPU. Cells labelled **Linux GPU lab** show the *real* vLLM-Omni commands — they are shown as text and are **not** run here, because the official `vllm serve --omni` runtime is CUDA-only.

## The OmniConnector decouples transport from model logic

Stages must hand each other embeddings, hidden states, audio codes, and image tensors. The **OmniConnector** generalizes vLLM's KV-cache transfer into one put/get interface over any transport:

- **single-node:** inline control queues for small payloads, shared memory for large ones
- **multi-node:** a Mooncake-based connector over TCP or RDMA, passing only lightweight metadata through the control plane

This is what makes **full disaggregation** possible: each stage runs on its own engine with its own accelerators, and the connector moves data between them.

In [ ]:
# Measured connector latencies from the paper (Qwen2.5-Omni), in milliseconds:
overhead = {
    'Thinker2Talker': {'shared_memory': 5.49, 'mooncake': 8.28},
    'Talker2Vocoder': {'shared_memory': 0.53, 'mooncake': 3.34},
}
for edge, modes in overhead.items():
    print(f"{edge:16} shared={modes['shared_memory']:.2f} ms  mooncake={modes['mooncake']:.2f} ms")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
edges = ['Thinker2Talker', 'Talker2Vocoder']
shared = [5.49, 0.53]; mooncake = [8.28, 3.34]
x = np.arange(len(edges)); w = 0.35
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.bar(x - w/2, shared, w, label='shared memory (1 node)')
ax.bar(x + w/2, mooncake, w, label='Mooncake (multi-node)')
ax.set_xticks(x); ax.set_xticklabels(edges)
ax.set_ylabel('transfer latency (ms)')
ax.set_title('Connector overhead by transport'); ax.legend()
fig.tight_layout(); plt.show()

## Overhead in context

End-to-end omni inference takes *tens of seconds*. A few milliseconds of transfer per edge is negligible — which is exactly why disaggregating stages across nodes is a good trade.

In [ ]:
inference_s = 20.0
worst_transfer_ms = 8.28 + 3.34
print(f'connector overhead is {worst_transfer_ms/1000/inference_s:.4%} of a 20s request')

## Exercise

If you scale the Talker onto a second node, which connector edge changes transport, and by how many ms? Compute the added latency.

In [ ]:
# Solution: Thinker->Talker crosses the node boundary and switches to Mooncake.
added = 8.28 - 5.49
print(f'Thinker2Talker goes shared->mooncake: +{added:.2f} ms (still trivial vs seconds)')

## Source lab

Find the connector interface in `vllm_omni/` and the shared-memory and Mooncake implementations. Locate the put/get calls and what metadata rides the control plane.